In [1]:
"""
generate.py
------------

Purpose:
1. Load the trained LSTM model.
2. Generate new music notes.
3. Convert generated notes into a MIDI file.
4. Save the generated music.
"""

import os
import random
import pickle
import numpy as np

from music21 import instrument
from music21 import note
from music21 import chord
from music21 import stream

from tensorflow.keras.models import load_model

# =====================================
# Paths
# =====================================

DATA_PATH = "/kaggle/input/datasets/omnia00nabil/"
MODEL_PATH = "/kaggle/input/notebooks/omnia00nabil/music-generation-train-model/models/music_model.keras"
OUTPUT_PATH = "/kaggle/working/generated_music"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# =====================================
# Load Model
# =====================================

print("Loading trained model...")

model = load_model(MODEL_PATH)

# =====================================
# Load Saved Data
# =====================================
notes = pickle.load(
    open(os.path.join(DATA_PATH, "notes-pkl/notes.pkl"), "rb")
)

pitchnames = pickle.load(
    open(os.path.join(DATA_PATH, "pitchnames/pitchnames.pkl"), "rb")
)

# Dictionary

note_to_int = dict(
    (note, number)
    for number, note in enumerate(pitchnames)
)

int_to_note = dict(
    (number, note)
    for number, note in enumerate(pitchnames)
)

# =====================================
# Prepare Seed Sequence
# =====================================

SEQUENCE_LENGTH = 100

start = random.randint(
    0,
    len(notes) - SEQUENCE_LENGTH - 1
)

pattern = [
    note_to_int[n]
    for n in notes[start:start + SEQUENCE_LENGTH]
]

generated_notes = []

print("Generating music...")

# =====================================
# Generate 500 Notes
# =====================================

for i in range(500):

    prediction_input = np.reshape(
        pattern,
        (1, SEQUENCE_LENGTH, 1)
    )

    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(
        prediction_input,
        verbose=0
    )[0]
    
    prediction = prediction / np.sum(prediction)
    
    index = np.random.choice(
        len(prediction),
        p=prediction
    )

    result = int_to_note[index]

    generated_notes.append(result)

    pattern.append(index)

    pattern = pattern[1:]

# =====================================
# Convert Notes to MIDI
# =====================================

offset = 0

output_notes = []

for pattern in generated_notes:

    # Chord

    if "." in pattern or pattern.isdigit():

        notes_in_chord = pattern.split(".")

        chord_notes = []

        for current_note in notes_in_chord:

            new_note = note.Note(int(current_note))

            new_note.storedInstrument = instrument.Piano()

            chord_notes.append(new_note)

        new_chord = chord.Chord(chord_notes)

        new_chord.offset = offset

        output_notes.append(new_chord)

    # Single Note

    else:

        new_note = note.Note(pattern)

        new_note.offset = offset

        new_note.storedInstrument = instrument.Piano()

        output_notes.append(new_note)

    offset += 0.5

# =====================================
# Save MIDI
# =====================================

midi_stream = stream.Stream(output_notes)

output_file = os.path.join(
    OUTPUT_PATH,
    "generated.mid"
)

midi_stream.write(
    "midi",
    fp=output_file
)

print("\nMusic Generated Successfully!")
print(output_file)

Loading trained model...


I0000 00:00:1784558248.314958      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784558248.320939      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Generating music...

Music Generated Successfully!
/kaggle/working/generated_music/generated.mid
